# Segmentación semantica utilizando U-Net

In [ ]:
import os
from glob import glob
import numpy as np
import matplotlib.pyplot as plt

import torch
from torch import nn, optim
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import v2 as transforms
from torchvision.io import decode_image
from torchvision import tv_tensors
from torchinfo import summary
from torchmetrics.segmentation import DiceScore

from tqdm import tqdm

datafolder = "datasets/fishes"

comprobamos que GPU tenemos disponible

In [ ]:
# revisar GPU disponible

if torch.cuda.is_available():
    str_device = "cuda"
elif torch.backends.mps.is_available():
    str_device = "mps"
elif torch.xpu.is_available():
    str_device = "xpu"
else:
    str_device = "cpu"

print(f"Acelerador {str_device} disponible")
device = torch.device(str_device)


## Definición de hiperparámetros

In [ ]:
# parametros de entrenamiento
lr = 1e-3
batch_size = 8
epochs = 5
image_size = (224, 224)

# aleatorización reproducible
seed = 1234
train_split = 0.8

## Carga de Imágenes

Para este caso, las imagenes estan en dos carpetas: Una con imagenes ya augmentadas y otras en estado sin augmentar. Para fines de prueba, utilizaremos la versión Augmentada

In [ ]:
data_dir = os.path.join(datafolder, "Fish_Dataset", "Fish_Dataset")
path_images = glob(os.path.join(data_dir, "**", "*.png"), recursive=True)
gt_images = [path for path in path_images if "GT" in path]
im_images = [path for path in path_images if "GT" not in path]
print(f"Imágenes de GT: {len(gt_images)}")
print(f"Imágenes de entrada: {len(im_images)}")

# Obtenemos los nombres de las clases a partir de las rutas de las imágenes no GT
clases = [
    os.path.basename(path)
    for path in glob(os.path.join(data_dir, "**", "*"))
    if "GT" not in path
]
print(clases)

In [ ]:
# vamos a aleatorizar las imagenes para separar en nuestro conjunto de entrenamiento y test
rng = np.random.default_rng(seed=seed)
idxs = rng.permutation(len(im_images))
train_idx = idxs[: int(train_split * len(idxs))]
val_idx = idxs[int(train_split * len(idxs)) :]

# Separamos las rutas de las imágenes de entrenamiento y validación
train_impaths = [im_images[i] for i in train_idx]
train_gtpaths = [gt_images[i] for i in train_idx]

val_impaths = [im_images[i] for i in val_idx]
val_gtpaths = [gt_images[i] for i in val_idx]

print(f"Imágenes de entrenamiento: {len(train_impaths)}")
print(f"Imágenes de validación: {len(val_impaths)}")

In [ ]:
# Armamos un objeto Dataset para cargar las imágenes y sus máscaras correspondientes
class FishDataset(Dataset):
    def __init__(self, inputs, targets, transforms=None):
        super().__init__()
        self.inputs = inputs
        self.targets = targets
        self.transforms = transforms

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        input_image = self.inputs[idx]
        target_image = self.targets[idx]

        image = decode_image(input_image, mode="RGB")
        image = tv_tensors.Image(image)
        mask = decode_image(target_image, mode="GRAY")
        mask = (mask > 128).float()  # Binarizamos la máscara
        mask = tv_tensors.Mask(mask)

        if self.transforms:
            image, mask = self.transforms(image, mask)

        return image, mask

In [ ]:
train_transforms = transforms.Compose(
    [
        transforms.Resize(image_size),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.RandomRotation(degrees=30),
        transforms.ToDtype(torch.float32, scale=True),
    ]
)

test_transforms = transforms.Compose(
    [
        transforms.Resize(image_size),
        transforms.ToDtype(torch.float32, scale=True),
    ]
)

In [ ]:
train_set = FishDataset(train_impaths, train_gtpaths, transforms=train_transforms)

valid_set = FishDataset(val_impaths, val_gtpaths, transforms=test_transforms)


In [ ]:
# vamos a probar una imagen y su máscara para verificar que se cargan correctamente
im, gt = train_set[0]
print(f"Imagen de entrada: {im.shape}, máscara: {gt.shape}")

fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax[0].imshow(im.permute(1, 2, 0))
ax[0].set_title("Imagen de entrada")
ax[0].axis("off")
ax[1].imshow(gt.squeeze(), cmap="gray")
ax[1].set_title("Máscara de GT")
ax[1].axis("off")
plt.show()

## Dataloaders
Pytorch para facilitar el proceso de mezclar y separar los datos, provee DataLoaders.
Los Dataloaders son objetos que permiten automatizar el muestreo de nuestros datos, generando nuestros batches de entrenamiento, mezclar los datos, paralelizar este proceso para acelerar el proceso de creacion de nuestras muestras, etc.

In [ ]:
train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(valid_set, batch_size=batch_size, shuffle=False)

## Modelo

El modelo U-Net es una red Neuronal convolucional diseñada para tareas de segmentación en imagenes biomédicas.
Se caracteriza por su forma de U, que contiene dos rutas:
- Ruta de encoding: Captura el contexto de la imagen de entrada, y contrae la información.
- Ruta de decoding: Sobremuestrea espacialmente y aplica convolucion para producir mapa de segmentación.

Una de sus particularidades, son sus conexiones entre los diferentes niveles de encoding/decoding. Esto permite comunicar mejor información a distintos niveles.

![UNet](assets/UNet.png)

**NOTA**: Esta versión incluye algunos elementos que la implementación original de Ronneberger no contiene, pero que actualmente son estandar.

- Uso de `BatchNorm` para estabilizar el entrenamiento.
- Desactivar sesgo (`bias=False`) en la `Conv2D`, al estar `BatchNorm`.
- Reemplazo de `nn.ConvTranspose2d` por `Upsample + Conv` para evitar artefactos de tablero de ajedrez 

In [ ]:
# vamos a definir algunas de las estructuras de UNet

## Convolución doble
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.conv(x)


## Downsampling
class Downsample(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = DoubleConv(in_channels, out_channels)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

    def forward(self, x):
        # Down va despues a la conexión
        # Mientras que pool va a la siguiente capa de downsampling
        down = self.conv(x)
        pool = self.pool(down)
        return down, pool


## Upsampling
class Upsample(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        # self.up = nn.ConvTranspose2d(
        #    in_channels, in_channels // 2, kernel_size=2, stride=2
        # )
        self.up = nn.Sequential(
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True),
            nn.Conv2d(in_channels, in_channels // 2, kernel_size=1),
        )
        self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x, skip):
        x = self.up(x)
        x = torch.cat((x, skip), dim=1)
        return self.conv(x)


In [ ]:
# UNet completo
class UNet(nn.Module):
    def __init__(self, in_channels=3, out_classes=1, features=[64, 128, 256, 512]):
        super().__init__()
        self.downs = nn.ModuleList()
        self.ups = nn.ModuleList()

        # Encoder
        for feature in features:
            self.downs.append(Downsample(in_channels, feature))
            in_channels = feature

        # Cuello de botella
        self.bottleneck = DoubleConv(features[-1], features[-1] * 2)

        # Decoder
        for feature in reversed(features):
            self.ups.append(Upsample(feature * 2, feature))

        self.final_conv = nn.Conv2d(
            in_channels=features[0], out_channels=out_classes, kernel_size=1
        )

    def forward(self, x):
        skips = []
        for down in self.downs:
            skip, x = down(x)
            skips.append(skip)

        x = self.bottleneck(x)

        for up, skip in zip(self.ups, reversed(skips)):
            x = up(x, skip)

        return self.final_conv(x)

In [ ]:
# lo instanciamos y vemos su estructura
modelo = UNet(in_channels=3, out_classes=1).to(device)

summary(
    modelo,
    input_size=(batch_size, 3, *image_size),
    col_names=["input_size", "output_size", "num_params", "trainable"],
    device=device,
)

In [ ]:
# Optimizador y función de pérdida
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(modelo.parameters(), lr=lr)

## Funciones de apoyo

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0

    dice_metric = DiceScore(1, include_background=True).to(device)

    with tqdm(total=len(loader), desc="Entrenando") as pbar:
        for images, masks in loader:
            images, masks = images.to(device), masks.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, masks)
            loss.backward()
            optimizer.step()

            # Actualizar métricas
            total_loss += loss.item()
            preds_binary = (outputs > 0.0).long()
            dice_metric.update(preds_binary, masks.long())

            # computamos para mostrar en la barra de progreso
            avg_loss = total_loss / (pbar.n + 1)
            avg_dice = dice_metric.compute().item()
            pbar.set_postfix_str(f"Loss: {avg_loss:.4f}, Dice: {avg_dice:.4f}")
            pbar.update(1)

    return total_loss / len(loader)


In [ ]:
def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0

    dice_metric = DiceScore(1, include_background=True).to(device)

    with torch.no_grad():
        for images, masks in tqdm(loader, desc="Evaluando"):
            images, masks = images.to(device), masks.to(device)

            outputs = model(images)
            loss = criterion(outputs, masks)
            # Actualizar métricas
            total_loss += loss.item()
            preds_binary = (outputs > 0.0).long()
            dice_metric.update(preds_binary, masks.long())

        dice = dice_metric.compute().item()
        print(f"[Eval] Loss: {total_loss / len(loader):.4f}, Dice: {dice:.4f}")

    return total_loss / len(loader)

In [ ]:
for epoch in range(epochs):
    print(f"Epoch {epoch + 1}/{epochs}")
    train_loss = train_epoch(modelo, train_loader, criterion, optimizer, device)
    val_loss = eval_epoch(modelo, val_loader, criterion, device)

In [ ]:
## Guardamos el modelo entrenado
torch.save(modelo.state_dict(), "saves/modelo_unet_fishes.pth")

## Evaluación
Para ver si el modelo logra segmentar, vamos a buscar una de las imagenes de test en su versión no augmentada para verificar que tan bien segmenta

In [ ]:
#

In [ ]:
non_augs_path = os.path.join(datafolder, "NA_Fish_Dataset", "**", "*")
non_augs_images = glob(non_augs_path, recursive=True)
print(f"Imágenes sin aumentos: {len(non_augs_images)}")
selected = np.random.choice(non_augs_images)
print(f"Imagen seleccionada: {selected}")

In [ ]:
# cargamos la imagen
image = decode_image(selected, mode="RGB")
image = tv_tensors.Image(image)
image = test_transforms(image.unsqueeze(0)).to(device)


In [ ]:
modelo.eval()
with torch.no_grad():
    output = modelo(image)
    pred_proba = torch.sigmoid(output)
    pred_mask = (pred_proba > 0.5).float().detach().cpu().numpy()

fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax[0].imshow(image.squeeze().permute(1, 2, 0).cpu())
ax[0].set_title("Imagen de entrada")
ax[0].axis("off")
ax[1].imshow(pred_mask.squeeze(), cmap="gray")
ax[1].set_title("Máscara predicha")
ax[1].axis("off")